In [ ]:
import slangpy as spy
import numpy as np
import quaternion
import pathlib
import matplotlib.pyplot as plt
from pyglm import glm
from PIL import Image
import jax
import jax.numpy as jnp

from bvhgs.camera import Camera
from bvhgs.renderer import Renderer
from bvhgs.gaussian import GaussianCloud
from bvhgs.data import SFMDataset

# Load a COLMAP DB

In [ ]:
sfm_dataset = SFMDataset()
sfm_dataset.load_from_colmap(
    colmap_path=pathlib.Path("../resources/dataset/tandt_db/tandt/truck/sparse/0/"),
    image_dir=pathlib.Path("../resources/dataset/tandt_db/tandt/truck/images/")
)

In [ ]:
camera, image_path = sfm_dataset[-1]
camera.to_slang()

In [ ]:
# Load the image.
image = Image.open(image_path)

In [ ]:
image_arr = np.array(image)
image_arr = image_arr.astype(np.float32) / 255
plt.imshow(image_arr)

# Load COLMAP Point Cloud

In [ ]:
gaussians = GaussianCloud()
gaussians.load_from_colmap(pathlib.Path("../resources/dataset/tandt_db/tandt/truck/sparse/0/"))
len(gaussians)

# Renderer

In [ ]:
render = Renderer(gaussians, camera)

In [ ]:
render.render()

In [ ]:
render_arr = render.render_target.to_numpy()[:, :, :3]
render_arr = np.flip(render_arr, axis=(0, 1))
plt.imshow(render_arr)

# Image Lnoss

In [ ]:
def image_loss(src: jnp.ndarray, dst: jnp.ndarray):
    return jnp.mean((dst - src)**2)

In [ ]:
image_loss(render_arr, image_arr)

In [ ]:
# Image grad.
loss_grad = jax.value_and_grad(image_loss)

In [ ]:
loss, render_target_grad = loss_grad(render_arr, image_arr)
plt.imshow(jnp.linalg.norm(render_target_grad, axis=-1))

## A Very Simple Gradient Descent

In [ ]:
lr = 5e4
num_epochs = 32

for i in range(num_epochs):
    loss, render_target_grad = loss_grad(render_arr, image_arr)
    render_arr -= render_target_grad * lr

plt.imshow(render_arr)